# W11C1 Lab: Prompting, Measured Instead of Guessed

Run every cell from the top. **Everything already works.**

Needs the local model running. If it is not, the notebook still runs
with placeholder replies so nothing breaks.

Today you will:

1. Score a zero-shot prompt on a real task, so you have a baseline.
2. Add examples and measure whether they actually helped.
3. Find the part of a prompt that matters most.

There is no test to run and nothing to submit. Each task tells you what
you should see when it is right.

In [ ]:
# Setup.
import re
import pandas as pd
import matplotlib.pyplot as plt

# Routing support tickets into three buckets. Deliberately NOT sentiment: the
# model already knows sentiment, so a prompt change would show nothing.
TESTS = [
    ("The parcel took three weeks to arrive.", "shipping"),
    ("It stopped working after a month.", "quality"),
    ("Far too expensive for what it is.", "price"),
    ("Box was dented when it got here.", "shipping"),
    ("The stitching came apart immediately.", "quality"),
    ("Half the cost of the competition.", "price"),
    ("Still waiting, no tracking update.", "shipping"),
    ("Solid metal, feels built to last.", "quality"),
]
LABELS = ["shipping", "quality", "price"]
print(f"{len(TESTS)} tickets to route into {LABELS}")

In [ ]:
# GIVEN. One helper for talking to the local model.
import ollama

MODEL = "qwen2.5:0.5b"
_offline_notice_shown = False

def ask(prompt, temperature=0.0, n=1):
    """Send a prompt to the local model. Returns a list of n replies.

    If Ollama is not running you get a fixed placeholder instead, so the
    notebook still executes end to end. Start it with:
        docker compose -f docker/docker-compose.yml up -d ollama
    """
    global _offline_notice_shown
    out = []
    for _ in range(n):
        try:
            r = ollama.chat(model=MODEL,
                            messages=[{"role": "user", "content": prompt}],
                            options={"temperature": temperature})
            out.append(r["message"]["content"].strip())
        except Exception:
            if not _offline_notice_shown:
                print("[no local model running: using placeholder replies]")
                _offline_notice_shown = True
            out.append("(placeholder)")
    return out

print("model:", MODEL)
print("test :", ask("Reply with exactly the word: ready")[0][:40])

## Part 1. A prompt is a thing you can score

Prompting feels like intuition until you put a number on it. Write the
prompt, run it over the test set, count what it got right.

In [ ]:
# GIVEN. Score any prompt template over the whole test set.
def evaluate(template, show=False):
    """template must contain {review}. Returns accuracy."""
    correct = 0
    rows = []
    for text, gold in TESTS:
        reply = ask(template.format(review=text))[0].lower()
        guess = next((l for l in LABELS if l in reply), "?")
        correct += guess == gold
        rows.append({"gold": gold, "guess": guess, "reply": reply[:38]})
    if show:
        print(pd.DataFrame(rows).to_string(index=False))
    return correct / len(TESTS)

ZERO_SHOT = "Categorise this customer review.\n\n{review}"
print("--- zero shot ---")
acc_zero = evaluate(ZERO_SHOT, show=True)
print(f"\naccuracy: {acc_zero:.2f}")

In [ ]:
# ================== YOUR TURN 1 ==================
# The bare prompt scores terribly: the model has no idea what
# categories you want, so it invents its own and the parser reads '?'.
#
# Tell it the three labels, and tell it to answer in one word.
#
# Expected: accuracy leaps, often from around 0.12 to roughly half. Naming the
#           label set is the single highest-value edit you can make to a prompt, and
#           it costs eight words. Most "the model is stupid" complaints are really
#           the model being asked an underspecified question.
# ===============================================
MY_PROMPT = """Categorise this customer review.

{review}"""          # <-- name the three labels, and ask for one word

acc = evaluate(MY_PROMPT, show=True)
print(f"\naccuracy: {acc:.2f}   (zero-shot baseline was {acc_zero:.2f})")

## Part 2. In-context learning: show, do not tell

Instead of describing the task, put a few solved examples in the prompt.
The model never updates a single weight, yet it gets better.

In [ ]:
# GIVEN. Few-shot: two worked examples, then the real question.
FEW_SHOT = """Categorise each customer review as shipping, quality, or price.

Review: Arrived a week late.
Category: shipping

Review: The handle snapped off.
Category: quality

Review: Cheaper than anywhere else.
Category: price

Review: {review}
Category:"""

acc_few = evaluate(FEW_SHOT)
print(f"zero shot : {acc_zero:.2f}")
print(f"few shot  : {acc_few:.2f}")

plt.figure(figsize=(4.6, 3))
plt.bar(["zero shot", "few shot"], [acc_zero, acc_few], color=["#999", "#7C2529"])
plt.ylim(0, 1); plt.ylabel("accuracy"); plt.title("Does showing examples help?")
plt.show()

In [ ]:
# ================== YOUR TURN 2 ==================
# How many examples do you actually need? Try 0, then 2, then 4.
#
# Compare each against the labelled zero-shot prompt from task 1.
#
# Expected: on a model this small, examples often help LESS than simply naming
#           the labels did, and the numbers move between runs because sampling is
#           noisy. Run each setting more than once before you believe a difference.
#           Examples cost tokens on every call, so measure before you pay.
# ===============================================
EXAMPLES = [
    ("Arrived a week late.", "shipping"),
    ("The handle snapped off.", "quality"),
    ("Cheaper than anywhere else.", "price"),
    ("Lost in transit twice.", "shipping"),
]

N_EXAMPLES = 2          # <-- try 0, then 4

shots = "".join(f"Review: {t}\nCategory: {c}\n\n" for t, c in EXAMPLES[:N_EXAMPLES])
prompt = ("Categorise each customer review as shipping, quality, or price.\n\n"
          + shots + "Review: {review}\nCategory:")

acc = evaluate(prompt)
print(f"{N_EXAMPLES} examples -> accuracy {acc:.2f}")
print(f"prompt length: {len(prompt)} characters")

## Answers

Try each task before reading.

In [ ]:
# YOUR TURN 1
#   Something like: "Categorise this review as shipping, quality, or price.
#   Answer with one word." Accuracy roughly quadruples. The bare prompt was
#   not a hard question, it was an UNDERSPECIFIED one.
#
# YOUR TURN 2
#   Examples help, but on a 0.5B model they often do NOT beat simply naming
#   the labels, and the gap moves between runs. Two lessons: the reflex to
#   reach for few-shot first is not always right, and a single run of eight
#   items cannot tell a real 0.12 difference from noise. Real evaluations use
#   hundreds of items for exactly this reason.